# InstructGPT (Ouyang et al., 2022) — Implementation / 구현

**Paper**: Ouyang et al. (2022), "Training Language Models to Follow Instructions with Human Feedback" (NeurIPS). arXiv:2203.02155.

## Overview / 개요

**English** This notebook implements the **3-step RLHF pipeline** at toy scale on a synthetic bandit-style text generation task:

1. **Reward Model (RM)**: a small neural network that scores `(prompt, completion)` pairs, trained with the **Bradley-Terry pairwise preference loss** on synthetic preference pairs.
2. **Reference policy (SFT analog)**: a fixed categorical distribution over actions per prompt; serves as the KL anchor.
3. **PPO-like RL update**: REINFORCE policy gradient with a **per-token KL penalty** $\beta \log(\pi_{RL} / \pi_{SFT})$ against the reference policy. We track reward, KL, and the effective objective $r - \beta \cdot \text{KL}$.

**한국어** 본 노트북은 **3단계 RLHF 파이프라인**을 합성 bandit 스타일 텍스트 생성 task에 toy 규모로 구현합니다:

1. **Reward Model (RM)**: `(prompt, completion)`을 스칼라 보상으로 매핑하는 소형 신경망. 합성 선호 쌍에 **Bradley-Terry pairwise loss**로 학습.
2. **Reference policy (SFT 대용)**: 각 prompt 별 고정 categorical 분포. KL anchor 역할.
3. **PPO 류 RL 업데이트**: per-token **KL penalty** $\beta \log(\pi_{RL} / \pi_{SFT})$를 더한 REINFORCE policy gradient. reward, KL, 그리고 유효 목적함수 $r - \beta \cdot \text{KL}$를 추적합니다.

Kernel: `study-with-ai`.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Part 1: Synthetic Preference Data / 합성 선호 데이터

**English** We construct a tiny world: 8 prompts, each with 6 candidate completions. Each completion is encoded as a 4-dim feature vector (a stand-in for a learned embedding). A hidden "true reward" function $r^\star(x, y) = w_x^\top \phi(y)$ generates ground-truth preferences. Synthetic labelers flip 5% of pairs (label noise).

**한국어** 작은 세계를 만듭니다: 8개 prompt, 각 prompt당 6개 후보 응답. 각 응답은 4차원 feature vector(임베딩 대용). 숨겨진 "true reward" $r^\star(x, y) = w_x^\top \phi(y)$가 ground-truth 선호를 생성. 합성 라벨러가 5% 쌍을 뒤집어 noise 추가.

In [ ]:
N_PROMPTS = 8
N_COMPLETIONS = 6        # number of candidate completions per prompt (K)
FEATURE_DIM = 4
LABEL_NOISE = 0.05

# Synthetic completion features: each completion is a 4-dim vector
completions = torch.randn(N_PROMPTS, N_COMPLETIONS, FEATURE_DIM)

# Ground-truth per-prompt reward weights (hidden from the RM)
true_w = torch.randn(N_PROMPTS, FEATURE_DIM)

# Ground-truth scalar reward for each (prompt, completion)
with torch.no_grad():
    true_reward = (completions * true_w.unsqueeze(1)).sum(-1)   # (N_PROMPTS, N_COMPLETIONS)

print('True reward per prompt (rows = prompts, cols = completions):')
print(true_reward.numpy().round(3))

# Build synthetic pairwise preferences: for each prompt, every (i, j) pair with i != j
# becomes a comparison; the higher true_reward wins (with 5% label noise).
pairs = []
for p in range(N_PROMPTS):
    for i in range(N_COMPLETIONS):
        for j in range(i + 1, N_COMPLETIONS):
            r_i, r_j = true_reward[p, i].item(), true_reward[p, j].item()
            winner, loser = (i, j) if r_i > r_j else (j, i)
            if np.random.rand() < LABEL_NOISE:
                winner, loser = loser, winner   # flip
            pairs.append((p, winner, loser))

print(f'\nGenerated {len(pairs)} preference pairs (from {N_PROMPTS} prompts x C(6,2) = 15 pairs each).')

## Part 2: Reward Model with Bradley-Terry Pairwise Loss / Bradley-Terry pairwise loss로 학습된 보상 모델

**English** The RM is a 2-layer MLP that takes a 4-dim completion feature plus a one-hot prompt id and outputs a scalar reward $r_\theta(x, y)$. We train with the InstructGPT loss:

$$\mathcal{L}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\bigl[\log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))\bigr]$$

**한국어** RM은 2-layer MLP로, 4차원 응답 feature + one-hot prompt id를 받아 스칼라 보상 출력. InstructGPT의 손실로 학습:

$$\mathcal{L}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\bigl[\log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))\bigr]$$

In [ ]:
class RewardModel(nn.Module):
    """Tiny RM: (prompt id one-hot, completion features) -> scalar reward."""

    def __init__(self, n_prompts: int, feat_dim: int, hidden: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_prompts + feat_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self.n_prompts = n_prompts

    def forward(self, prompt_ids: torch.Tensor, feats: torch.Tensor) -> torch.Tensor:
        """Compute scalar reward.

        Args:
            prompt_ids: Long tensor of shape (B,) with prompt indices.
            feats: Float tensor of shape (B, feat_dim) with completion features.

        Returns:
            Scalar reward tensor of shape (B,).
        """
        one_hot = F.one_hot(prompt_ids, num_classes=self.n_prompts).float()
        x = torch.cat([one_hot, feats], dim=-1)
        return self.net(x).squeeze(-1)


rm = RewardModel(N_PROMPTS, FEATURE_DIM).to(device)
rm_opt = torch.optim.Adam(rm.parameters(), lr=1e-2)

# Convert pairs to tensors
pair_tensor = torch.tensor(pairs, dtype=torch.long)              # (N_pairs, 3)
completions_dev = completions.to(device)

rm_losses, rm_accs = [], []
for epoch in range(400):
    # Shuffle
    perm = torch.randperm(len(pair_tensor))
    batch = pair_tensor[perm].to(device)
    p_idx, w_idx, l_idx = batch[:, 0], batch[:, 1], batch[:, 2]

    feats_w = completions_dev[p_idx, w_idx]
    feats_l = completions_dev[p_idx, l_idx]
    r_w = rm(p_idx, feats_w)
    r_l = rm(p_idx, feats_l)

    # Bradley-Terry pairwise loss
    loss = -F.logsigmoid(r_w - r_l).mean()

    rm_opt.zero_grad()
    loss.backward()
    rm_opt.step()

    acc = ((r_w > r_l).float()).mean().item()
    rm_losses.append(loss.item())
    rm_accs.append(acc)

print(f'Final RM loss = {rm_losses[-1]:.4f}, accuracy on training pairs = {rm_accs[-1]:.3f}')

# Plot training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(rm_losses); ax[0].set(title='RM Bradley-Terry loss', xlabel='epoch', ylabel='loss')
ax[1].plot(rm_accs); ax[1].set(title='RM pairwise accuracy', xlabel='epoch', ylabel='accuracy')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Sanity check: does the RM agree with the true reward? / 정합성 검증

**English** We compare the learned reward to the (hidden) true reward via Spearman rank correlation per prompt. A well-trained RM should rank completions almost identically to ground truth.

**한국어** 학습된 RM이 (숨겨진) true reward와 같은 랭킹을 만드는지 prompt별 Spearman 상관계수로 확인. 잘 학습된 RM은 ground truth와 거의 같은 랭킹을 만들어야 합니다.

In [ ]:
from scipy.stats import spearmanr

with torch.no_grad():
    learned = torch.zeros_like(true_reward)
    for p in range(N_PROMPTS):
        p_idx = torch.full((N_COMPLETIONS,), p, dtype=torch.long, device=device)
        learned[p] = rm(p_idx, completions_dev[p]).cpu()

rho_per_prompt = []
for p in range(N_PROMPTS):
    rho, _ = spearmanr(true_reward[p].numpy(), learned[p].numpy())
    rho_per_prompt.append(rho)

print('Per-prompt Spearman rank correlation (RM vs ground truth):')
for p, r in enumerate(rho_per_prompt):
    print(f'  prompt {p}: rho = {r:+.3f}')
print(f'\nMean rho = {np.mean(rho_per_prompt):.3f}')

## Part 3: Reference Policy (SFT analog) / 참조 정책 (SFT 대용)

**English** In InstructGPT, the SFT model is the KL anchor. Here we mimic an SFT model with a fixed categorical distribution per prompt — the **uniform** distribution biased slightly toward the median completion. This pretends to be "what humans demonstrate", so the RL policy is penalized for drifting from it.

**한국어** InstructGPT에서 SFT 모델이 KL anchor입니다. 여기서는 prompt별 고정 categorical 분포로 SFT를 모방합니다 — 약간 median completion 쪽으로 bias된 거의 uniform한 분포. 이는 "인간이 시범한 것"을 흉내내며, RL 정책이 이로부터 멀어지면 페널티 받습니다.

In [ ]:
# Reference (SFT) policy: per-prompt categorical distribution
# Slight bias toward 'medium-ranked' completions (mimicking human demonstrations
# that are decent but not optimal under the unknown reward).
with torch.no_grad():
    sft_logits = torch.zeros(N_PROMPTS, N_COMPLETIONS)
    for p in range(N_PROMPTS):
        order = torch.argsort(true_reward[p], descending=True)
        # Highest weight to ranks 1-2 (decent), lowest to rank 0 (best)
        rank_weights = torch.tensor([0.05, 0.30, 0.30, 0.20, 0.10, 0.05])
        sft_logits[p, order] = torch.log(rank_weights + 1e-8)

sft_logprobs = F.log_softmax(sft_logits, dim=-1).to(device)
sft_probs = sft_logprobs.exp()

print('Reference (SFT) policy probabilities (rows = prompts, cols = completions):')
print(sft_probs.cpu().numpy().round(3))
print(f'\nReference policy expected reward = {(sft_probs.cpu() * true_reward).sum(-1).mean().item():+.3f}')
print(f'Optimal expected reward = {true_reward.max(-1).values.mean().item():+.3f}')

## Part 4: KL-Regularized Policy Gradient (PPO-style toy update) / KL-정규화 정책 경사

**English** We parameterize the RL policy as per-prompt logits $\theta_{p,k}$. At each step we sample a completion, compute the RM reward, and the KL-penalized objective

$$\text{obj} = r_\theta(x, y) - \beta \, \log \frac{\pi_{RL}(y | x)}{\pi_{SFT}(y | x)}$$

We use REINFORCE with a moving-average baseline. PPO's clipped surrogate is omitted for clarity—the conceptual essentials (reward + KL anchor) are the same.

**한국어** RL 정책을 prompt 별 logit $\theta_{p,k}$로 파라미터화. 매 step에서 completion을 sampling, RM reward 계산, KL-penalized 목적함수

$$\text{obj} = r_\theta(x, y) - \beta \, \log \frac{\pi_{RL}(y | x)}{\pi_{SFT}(y | x)}$$

moving-average baseline을 사용한 REINFORCE로 학습. PPO의 clipped surrogate은 생략 — 개념적 핵심(reward + KL anchor)은 동일.

In [ ]:
BETA_KL = 0.5            # KL penalty coefficient
LR_RL = 0.05
STEPS = 600
BATCH_PROMPTS = 32       # prompts per RL step (sampled with replacement)

# Initialize RL policy = reference policy (so KL starts at 0)
rl_logits = sft_logits.clone().to(device).requires_grad_(True)
rl_opt = torch.optim.Adam([rl_logits], lr=LR_RL)

history = {'reward': [], 'kl': [], 'objective': []}
baseline = 0.0

for step in range(STEPS):
    # Sample prompts
    p_idx = torch.randint(0, N_PROMPTS, (BATCH_PROMPTS,), device=device)

    # Current RL policy distribution for sampled prompts
    logits_b = rl_logits[p_idx]
    rl_logp_b = F.log_softmax(logits_b, dim=-1)
    rl_p_b = rl_logp_b.exp()

    # Sample completions
    dist = torch.distributions.Categorical(probs=rl_p_b)
    a = dist.sample()                                     # (BATCH,)
    logp_rl_a = rl_logp_b.gather(1, a.unsqueeze(1)).squeeze(1)
    logp_sft_a = sft_logprobs[p_idx].gather(1, a.unsqueeze(1)).squeeze(1)

    # RM reward (no grad through the RM)
    feats = completions_dev[p_idx, a]
    with torch.no_grad():
        r = rm(p_idx, feats)

    # Per-token KL penalty (here per-action; sequence length = 1)
    kl_token = logp_rl_a - logp_sft_a                      # (BATCH,)
    shaped_reward = r - BETA_KL * kl_token.detach()

    # REINFORCE with moving-average baseline
    advantage = shaped_reward - baseline
    pg_loss = -(logp_rl_a * advantage.detach()).mean()

    rl_opt.zero_grad()
    pg_loss.backward()
    rl_opt.step()

    # Diagnostics: full-distribution KL and expected reward
    with torch.no_grad():
        full_kl = (rl_p_b * (rl_logp_b - sft_logprobs[p_idx])).sum(-1).mean().item()
        ground_truth_r = true_reward.to(device)[p_idx, a].mean().item()
        rm_r = r.mean().item()
        obj = (rm_r - BETA_KL * full_kl)

    history['reward'].append(ground_truth_r)
    history['kl'].append(full_kl)
    history['objective'].append(obj)

    # Update baseline (EMA)
    baseline = 0.95 * baseline + 0.05 * shaped_reward.mean().item()

print(f'Step {STEPS}: ground-truth reward = {history["reward"][-1]:+.3f}, '
      f'KL(RL || SFT) = {history["kl"][-1]:.3f}, '
      f'objective = {history["objective"][-1]:+.3f}')

## Part 5: Diagnostics & the Effect of $\beta$ / 진단과 $\beta$의 영향

**English** We plot the training curves to verify the expected RLHF behavior:
- Ground-truth reward should increase as the policy learns to pick high-reward completions.
- KL(RL || SFT) should rise from 0 as the policy diverges from the reference.
- The KL-penalized objective should monotonically improve.

Then we sweep $\beta \in \{0, 0.1, 0.5, 2.0\}$ to show the **alignment-tax / reward-hacking trade-off**: $\beta = 0$ achieves highest RM reward but largest KL (often hacks); large $\beta$ keeps the policy near SFT (low ground-truth reward but safe).

**한국어** 학습 곡선을 그려 RLHF의 기대 행동을 확인:
- Ground-truth reward 증가.
- KL(RL || SFT)는 0에서 증가.
- KL-페널티 목적함수는 단조 증가.

그 다음 $\beta \in \{0, 0.1, 0.5, 2.0\}$ sweep으로 **alignment tax / reward hacking trade-off**를 시각화: $\beta = 0$은 RM reward 최대지만 KL 폭발 (hacking), 큰 $\beta$는 SFT 근처에 머물러 ground-truth reward가 낮음.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(history['reward']); ax[0].set(title='Ground-truth reward (smoothed below)',
                                          xlabel='step', ylabel='reward')
ax[1].plot(history['kl']); ax[1].set(title=r'KL($\pi_{RL}$ || $\pi_{SFT}$)',
                                      xlabel='step', ylabel='KL')
ax[2].plot(history['objective']); ax[2].set(title=r'$r - \beta \cdot $ KL  (objective)',
                                              xlabel='step', ylabel='objective')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
def train_rlhf(beta: float, steps: int = 600) -> dict:
    """Run KL-regularized REINFORCE for a given KL coefficient beta.

    Args:
        beta: KL penalty coefficient.
        steps: number of RL update steps.

    Returns:
        Dict with final ground-truth reward, RM reward, and KL.
    """
    logits = sft_logits.clone().to(device).requires_grad_(True)
    opt = torch.optim.Adam([logits], lr=LR_RL)
    bl = 0.0
    for _ in range(steps):
        p_idx = torch.randint(0, N_PROMPTS, (BATCH_PROMPTS,), device=device)
        logits_b = logits[p_idx]
        logp = F.log_softmax(logits_b, dim=-1)
        p = logp.exp()
        a = torch.distributions.Categorical(probs=p).sample()
        logp_a = logp.gather(1, a.unsqueeze(1)).squeeze(1)
        logp_sft_a = sft_logprobs[p_idx].gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            r = rm(p_idx, completions_dev[p_idx, a])
        kl_token = logp_a - logp_sft_a
        shaped = r - beta * kl_token.detach()
        adv = shaped - bl
        loss = -(logp_a * adv.detach()).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        bl = 0.95 * bl + 0.05 * shaped.mean().item()

    # Final evaluation: expected metrics under the learned policy
    with torch.no_grad():
        all_logp = F.log_softmax(logits, dim=-1)
        all_p = all_logp.exp()
        full_kl = (all_p * (all_logp - sft_logprobs)).sum(-1).mean().item()
        # Expected RM reward
        rm_r = 0.0
        for p_id in range(N_PROMPTS):
            r_all = rm(torch.full((N_COMPLETIONS,), p_id, dtype=torch.long, device=device),
                       completions_dev[p_id])
            rm_r += (all_p[p_id] * r_all).sum().item()
        rm_r /= N_PROMPTS
        gt_r = (all_p.cpu() * true_reward).sum(-1).mean().item()

    return {'beta': beta, 'gt_reward': gt_r, 'rm_reward': rm_r, 'kl': full_kl}


betas = [0.0, 0.1, 0.5, 2.0]
results = [train_rlhf(b) for b in betas]

print(f'{"beta":>5} | {"GT reward":>10} | {"RM reward":>10} | {"KL(RL||SFT)":>12}')
print('-' * 50)
for r in results:
    print(f'{r["beta"]:>5.2f} | {r["gt_reward"]:>+10.3f} | {r["rm_reward"]:>+10.3f} | {r["kl"]:>12.3f}')

betas_arr = np.array([r['beta'] for r in results])
kls = np.array([r['kl'] for r in results])
rm_rs = np.array([r['rm_reward'] for r in results])
gt_rs = np.array([r['gt_reward'] for r in results])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(betas_arr, kls, 'o-', color='tab:blue', label='KL(RL || SFT)')
ax[0].set(xlabel=r'$\beta$', ylabel='KL', title='KL vs beta (alignment-tax knob)'); ax[0].grid(alpha=0.3)
ax[1].plot(betas_arr, rm_rs, 'o-', color='tab:orange', label='RM reward')
ax[1].plot(betas_arr, gt_rs, 's-', color='tab:green', label='Ground-truth reward')
ax[1].set(xlabel=r'$\beta$', ylabel='reward', title='Reward vs beta'); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()

## Summary / 요약

| Concept / 개념 | This Notebook / 본 노트북 | InstructGPT (Ouyang et al. 2022) |
|---|---|---|
| Reward Model | 2-layer MLP, 4-dim features | 6B Transformer with scalar head from SFT |
| RM training data | 8 prompts × 15 pairs (synthetic) | 33K prompts × $\binom{K}{2}$ pairs from 40 contractors |
| RM loss | Bradley-Terry pairwise (identical formula) | Bradley-Terry pairwise (identical formula) |
| Reference policy | Fixed categorical, biased toward median | 175B SFT model fine-tuned on demonstrations |
| RL algorithm | REINFORCE + EMA baseline | PPO (clipped surrogate + advantage) |
| KL anchor | $\beta \log(\pi_{RL}/\pi_{SFT})$ per action | $\beta \log(\pi_{RL}/\pi_{SFT})$ per token |
| Pretraining mix ($\gamma$) | omitted (no pretraining data) | $\gamma \approx 27.8$ for PPO-ptx |
| Eval | ground-truth synthetic reward | labeler win-rate, TruthfulQA, RealToxicityPrompts |

**English Key takeaways from this toy implementation**:
1. The Bradley-Terry pairwise loss reliably recovers a reward function consistent with the latent ground truth.
2. RLHF without a KL anchor ($\beta = 0$) achieves highest RM reward but inflates KL — empirical reward-hacking signature.
3. Increasing $\beta$ shrinks KL but at the cost of ground-truth reward — the alignment tax knob.
4. The optimal $\beta$ balances RM exploitation with proximity to the SFT prior, exactly as in InstructGPT (where $\beta \approx 0.02$ at full scale).

**한국어 핵심 결론**:
1. Bradley-Terry pairwise loss는 잠재 ground-truth와 일치하는 reward function을 안정적으로 복원합니다.
2. KL anchor 없이 RLHF ($\beta = 0$)는 RM reward를 최대화하지만 KL이 폭증 — reward hacking의 경험적 지표.
3. $\beta$ 증가는 KL을 줄이지만 ground-truth reward를 희생 — alignment tax knob.
4. 최적 $\beta$는 RM 익스플로잇과 SFT prior 근접성의 균형 — InstructGPT의 $\beta \approx 0.02$도 같은 원리.